In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from ufc_scraper.utils import read_fight_summary


In [2]:
fight_summary = read_fight_summary().collect()

In [ ]:
fight_summary.head()

In [ ]:
FIGHT_SUMMARY_FIGHTER_COLS = [
    "result",
    "nickname",
]

FIGHT_SUMMARY_BASE_COLS = [
    "fight_id",
    "event_id",
    "event_name",
    "bout_type",
    "method",
    "round",
    "time",
    "time_seconds",
    "time_format",
    "referee",
    "details",
    "scraped_at",
    "ingested_at",
]

In [ ]:
def transform_fight_summary(
    fight_summary: pl.DataFrame,
) -> pl.DataFrame:

    return (
        fight_summary
        .with_columns(

            pl.col("round")
            .cast(pl.Int64),

            (
                pl.col("time")
                .str.extract(r"(\d+):(\d+)", 1)
                .cast(pl.Int64)
                * 60
                +
                pl.col("time")
                .str.extract(r"(\d+):(\d+)", 2)
                .cast(pl.Int64)
            ).alias("time_seconds"),

            pl.col("scraped_at")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("ingested_at")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("fighter_1").str.strip_chars(),
            pl.col("fighter_2").str.strip_chars(),

            pl.col("fighter_1_nickname")
            .str.strip_chars('"'),

            pl.col("fighter_2_nickname")
            .str.strip_chars('"'),
        )
    )

In [ ]:
def build_fight_summary_fighter_view(
    df: pl.DataFrame,
    fighter_num: int,
) -> pl.DataFrame:

    other = 2 if fighter_num == 1 else 1

    exprs = [
        pl.col(f"fighter_{fighter_num}").alias("fighter"),
        pl.col(f"fighter_{other}").alias("opponent"),

        pl.col(f"fighter_{fighter_num}_result")
        .alias("fighter_result"),

        pl.col(f"fighter_{fighter_num}_nickname")
        .alias("nickname"),
    ]

    return df.select(
        *FIGHT_SUMMARY_BASE_COLS,
        "winner",
        "loser",
        *exprs,
    )

In [ ]:
def create_fight_summary_long(
    fight_summary: pl.DataFrame,
) -> pl.DataFrame:

    fighter1_df = build_fight_summary_fighter_view(
        fight_summary,
        1,
    )

    fighter2_df = build_fight_summary_fighter_view(
        fight_summary,
        2,
    )

    return pl.concat(
        [fighter1_df, fighter2_df],
        how="vertical",
    )

In [ ]:
fight_summary_long = (
    fight_summary
    .pipe(transform_fight_summary)
    .pipe(create_fight_summary_long)
)


In [ ]:
fight_summary_long.shape

In [ ]:
fight_summary.shape

In [ ]:
fight_summary_long

In [ ]:
fight_summary_long.filter(pl.col('fight_id') == 'fb77b08a90b92d5b')

In [ ]:
fight_summary.filter(pl.col('fight_id') == 'fb77b08a90b92d5b')

### New Pipeline

In [ ]:
import polars as pl


FIGHT_SUMMARY_COLS = [
    "fight_id",
    "event_id",
    "event_name",
    "fighter_1",
    "fighter_2",
    "winner",
    "loser",
    "bout_type",
    "method",
    "round",
    "time",
    "time_seconds",
    "referee",
    "details",
    "scraped_at",
    "ingested_at",
]


def transform_fight_summary(
    fight_summary: pl.DataFrame,
) -> pl.DataFrame:

    return (
        fight_summary
        .with_columns(

            pl.col("round")
            .cast(pl.Int64),

            (
                pl.col("time")
                .str.extract(r"(\d+):(\d+)", 1)
                .cast(pl.Int64)
                * 60
                +
                pl.col("time")
                .str.extract(r"(\d+):(\d+)", 2)
                .cast(pl.Int64)
            ).alias("time_seconds"),

            pl.col("scraped_at")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("ingested_at")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("fighter_1").str.strip_chars(),

            pl.col("fighter_2").str.strip_chars(),

            pl.col("winner").str.strip_chars(),

            pl.col("loser").str.strip_chars(),

            pl.col("event_name").str.strip_chars(),

            pl.col("bout_type").str.strip_chars(),

            pl.col("method").str.strip_chars(),

            pl.col("referee").str.strip_chars(),
        )
        .select(FIGHT_SUMMARY_COLS)
    )


In [ ]:
fight_summary_clean = (
    fight_summary
    .pipe(transform_fight_summary)
)

In [ ]:
fight_summary_clean.shape

In [ ]:
fight_summary_clean

In [ ]:
fight_summary

### New Pipeline 2

In [15]:
import polars as pl


def standardize_dataframe(
    df: pl.DataFrame,
    empty_to_null: bool = False,
) -> pl.DataFrame:
    """
    Standardize a Polars DataFrame.

    - Strips whitespace from column names.
    - Strips whitespace from all string values.
    - Removes surrounding double quotes from string values.
    - Optionally converts empty strings to nulls.
    """

    # Clean column names
    df = df.rename(
        {
            col: col.strip()
            for col in df.columns
        }
    )

    string_cols = [
        col
        for col, dtype in df.schema.items()
        if dtype == pl.String
    ]

    exprs = []

    for col in string_cols:
        expr = (
            pl.col(col)
            .str.strip_chars()
            .str.strip_chars('"')
        )

        if empty_to_null:
            expr = (
                pl.when(expr == "")
                .then(None)
                .otherwise(expr)
            )

        exprs.append(expr.alias(col))

    return df.with_columns(*exprs)

In [16]:
import polars as pl


FIGHT_SUMMARY_COLS = [
    "fight_id",
    "event_id",
    "event_name",
    "fighter_1",
    "fighter_2",
    "fight_result",
    "bout_type",
    "method",
    "round",
    "time",
    "time_seconds",
    "referee",
    "judge_1_score",
    "judge_2_score",
    "judge_3_score",
    "details",
    "scraped_at",
    "ingested_at",
]


def transform_fight_summary(
    fight_summary: pl.DataFrame,
) -> pl.DataFrame:

    score_expr = pl.col("details").str.extract_all(
        r"\d+\s*-\s*\d+"
    )

    return (
        fight_summary
        .with_columns(

            # Round
            pl.col("round")
            .cast(pl.Int64),

            # Fight duration in seconds
            (
                pl.col("time")
                .str.extract(r"(\d+):(\d+)", 1)
                .cast(pl.Int64)
                * 60
                +
                pl.col("time")
                .str.extract(r"(\d+):(\d+)", 2)
                .cast(pl.Int64)
            ).alias("time_seconds"),

            # Datetime columns
            pl.col("scraped_at")
            .str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S%z"
            ),

            pl.col("ingested_at")
            .str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S%z"
            ),

            # Clean text
            pl.col("fighter_1").str.strip_chars(),
            pl.col("fighter_2").str.strip_chars(),
            pl.col("winner").str.strip_chars(),
            pl.col("event_name").str.strip_chars(),
            pl.col("bout_type").str.strip_chars(),
            pl.col("method").str.strip_chars(),
            pl.col("referee").str.strip_chars(),

            # Fight result
            pl.when(
                pl.col("winner") == pl.col("fighter_1")
            )
            .then(pl.lit("FIGHTER_1_WIN"))

            .when(
                pl.col("winner") == pl.col("fighter_2")
            )
            .then(pl.lit("FIGHTER_2_WIN"))

            .when(
                pl.col("result")
                .str.to_lowercase()
                .eq("draw")
            )
            .then(pl.lit("DRAW"))

            .when(
                pl.col("result")
                .str.to_lowercase()
                .eq("no contest")
            )
            .then(pl.lit("NO_CONTEST"))

            .otherwise(pl.lit("UNKNOWN"))
            .alias("fight_result"),

            # Judge scores
            pl.when(
                pl.col("method").str.contains("Decision")
            )
            .then(
                score_expr.list.get(
                    0,
                    null_on_oob=True,
                )
            )
            .otherwise(None)
            .alias("judge_1_score"),

            pl.when(
                pl.col("method").str.contains("Decision")
            )
            .then(
                score_expr.list.get(
                    1,
                    null_on_oob=True,
                )
            )
            .otherwise(None)
            .alias("judge_2_score"),

            pl.when(
                pl.col("method").str.contains("Decision")
            )
            .then(
                score_expr.list.get(
                    2,
                    null_on_oob=True,
                )
            )
            .otherwise(None)
            .alias("judge_3_score"),
        )
        .select(FIGHT_SUMMARY_COLS)
    )


fight_summary_clean = (
    fight_summary
    .pipe(standardize_dataframe, empty_to_null=True)
    .pipe(transform_fight_summary)
)

In [17]:
fight_summary_clean

fight_id,event_id,event_name,fighter_1,fighter_2,fight_result,bout_type,method,round,time,time_seconds,referee,judge_1_score,judge_2_score,judge_3_score,details,scraped_at,ingested_at
str,str,str,str,str,str,str,str,i64,str,i64,str,str,str,str,str,"datetime[μs, UTC]","datetime[μs, UTC]"
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Anthony Hernandez""","""Gregory Rodrigues""","""FIGHTER_2_WIN""","""Middleweight Bout""","""Decision - Unanimous""",5,"""5:00""",300,"""Herb Dean""","""47 - 48""","""46 - 49""","""47 - 48""","""Derek Cleary 47 - 48. Chris Le…",2026-09-03 12:54:29 UTC,2026-09-03 13:23:47 UTC
"""a57ccf31c5f6f82b""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Serghei Spivac""","""Vitor Petrino""","""FIGHTER_2_WIN""","""Heavyweight Bout""","""Decision - Unanimous""",3,"""5:00""",300,"""Mike Beltran""","""27 - 30""","""27 - 30""","""27 - 30""","""Derek Cleary 27 - 30. Ron McCa…",2026-09-03 12:54:42 UTC,2026-09-03 13:23:47 UTC
"""0eba299c9b6760fa""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Reinier de Ridder""","""Roman Dolidze""","""FIGHTER_1_WIN""","""Light Heavyweight Bout""","""KO/TKO""",1,"""4:01""",241,"""Frank Trigg""",null,null,null,"""Punches to Head From Back Cont…",2026-09-03 12:54:54 UTC,2026-09-03 13:23:47 UTC
"""bf70d01518c80fba""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""MarQuel Mederos""","""Mason Jones""","""FIGHTER_1_WIN""","""Lightweight Bout""","""KO/TKO""",2,"""2:07""",127,"""Jason Herzog""",null,null,null,"""Punches to Head At Distance""",2026-09-03 12:55:04 UTC,2026-09-03 13:23:47 UTC
"""a4b3becc0cc394d7""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Carli Judice""","""Jeisla Chaves""","""FIGHTER_1_WIN""","""Women's Flyweight Bout""","""KO/TKO""",1,"""1:39""",99,"""David Shurley""",null,null,null,"""Kick to Body At Distance""",2026-09-03 12:55:15 UTC,2026-09-03 13:23:47 UTC
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""bbd05484f8c8d739""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Chris Padilla""","""Nasrat Haqparast""","""FIGHTER_1_WIN""","""Lightweight Bout""","""Submission""",3,"""4:59""",299,"""Frank Trigg""",null,null,null,"""Arm Triangle Standing""",2026-09-03 12:55:59 UTC,2026-09-03 13:23:47 UTC
"""7e86f774f19e5e59""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Marcio Barbosa""","""Ryan Kuse""","""FIGHTER_1_WIN""","""Featherweight Bout""","""KO/TKO""",1,"""2:47""",167,"""Jason Herzog""",null,null,null,"""Punch to Head At Distance""",2026-09-03 12:56:11 UTC,2026-09-03 13:23:47 UTC
"""116dc88b63113f61""","""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…","""Gauge Young""","""Stan Dorsainvil""","""FIGHTER_2_WIN""","""Lightweight Bout""","""Decision - Unanimous""",3,"""5:00""",300,"""Mike Beltran""","""28 - 29""","""28 - 29""","""28 - 29""","""Daniel Mobley 28 - 29. Ron McC…",2026-09-03 12:56:23 UTC,2026-09-03 13:23:47 UTC
